<a href="https://colab.research.google.com/github/wendydonzse/github-portfolio/blob/main/run_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch scikit-learn pandas numpy matplotlib seaborn -q

In [ ]:
from google.colab import files
import os

os.makedirs('/content/data/raw', exist_ok=True)

print("Upload your JSON files now:")
uploaded = files.upload()

for fname, data in uploaded.items():
    dest = f'/content/data/raw/{fname}'
    with open(dest, 'wb') as f:
        f.write(data)
    print(f"✅ Saved → {dest}")

print("\nFiles in /content/data/raw/:")
print(os.listdir('/content/data/raw'))

Upload your JSON files now:


Saving DEV_TEST_Merged_ASQE_N2500.json to DEV_TEST_Merged_ASQE_N2500.json
Saving Train_validation_Merged_ASQE_N4000.json to Train_validation_Merged_ASQE_N4000.json
✅ Saved → /content/data/raw/DEV_TEST_Merged_ASQE_N2500.json
✅ Saved → /content/data/raw/Train_validation_Merged_ASQE_N4000.json

Files in /content/data/raw/:
['Train_validation_Merged_ASQE_N4000.json', 'DEV_TEST_Merged_ASQE_N2500.json']


In [ ]:
%%writefile /content/config.py
import os
import random
import numpy as np
import torch

# ── Paths ──────────────────────────────────────────────────
BASE_DIR        = '/content'
DATA_DIR        = os.path.join(BASE_DIR, 'data')
RAW_DATA_DIR    = os.path.join(DATA_DIR, 'raw')
PROCESSED_DATA_DIR = os.path.join(DATA_DIR, 'processed')
MODEL_DIR       = os.path.join(BASE_DIR, 'models')
RESULTS_DIR     = os.path.join(BASE_DIR, 'results')

# ── Raw data filenames ─────────────────────────────────────
TRAIN_VAL_FILE  = 'Train_validation_Merged_ASQE_N4000.json'
DEV_TEST_FILE   = 'DEV_TEST_Merged_ASQE_N2500.json'

# ── Processed data paths ───────────────────────────────────
TRAIN_AUG_PATH  = os.path.join(PROCESSED_DATA_DIR, 'train_augmented.csv')
VAL_CLEAN_PATH  = os.path.join(PROCESSED_DATA_DIR, 'val_clean.csv')
TEST_CLEAN_PATH = os.path.join(PROCESSED_DATA_DIR, 'test_clean.csv')

# ── Results paths ──────────────────────────────────────────
FINAL_METRICS_PATH = os.path.join(RESULTS_DIR, 'final_summary_metrics.csv')

# ── Label config ───────────────────────────────────────────
TARGET_COLUMN   = 'sentiment'
LABEL_MAPPING   = {'Positive': 0, 'Negative': 1, 'Neutral': 2}
ID2LABEL        = {0: 'Positive', 1: 'Negative', 2: 'Neutral'}

# ── Model config ───────────────────────────────────────────
CANDIDATE_MODELS = [
    'bert-base-uncased',
    'roberta-base',
    'distilbert-base-uncased'
]
SELECTED_MODEL  = 'bert-base-uncased'   # ← update after Stage 2

# ── Training config ────────────────────────────────────────
MAX_SEQ_LENGTH  = 128
BATCH_SIZE      = 16
NUM_EPOCHS      = 3
LEARNING_RATE   = 2e-5
WEIGHT_DECAY    = 0.01
RANDOM_SEED     = 42

# ── Tuning grid ────────────────────────────────────────────
TUNING_PARAM_GRID = {
    'learning_rate': [2e-5, 3e-5],
    'batch_size':    [16, 32],
    'weight_decay':  [0.01, 0.001]
}

# ── Cross-validation ───────────────────────────────────────
N_FOLDS         = 5

# ── Create dirs ────────────────────────────────────────────
for d in [RAW_DATA_DIR, PROCESSED_DATA_DIR, MODEL_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

def set_global_seed(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print("✅ config.py loaded")

Writing /content/config.py


In [ ]:
# ── Stage 1 ────────────────────────────────────────────────

# [paste your full 01_data_pipeline.py code here]
%%writefile /content/01_data_pipeline.py
import os
import sys
import json
import logging
import importlib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

sys.path.insert(0, '/content')
if 'config' in sys.modules:
    importlib.reload(sys.modules['config'])
import config

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")


class DataPipeline:

    def __init__(self):
        self.raw_dir  = config.RAW_DATA_DIR
        self.proc_dir = config.PROCESSED_DATA_DIR

    # ----------------------------------------------------------
    # Load JSON files
    # ----------------------------------------------------------
    def _load_json(self, filename):
        path = os.path.join(self.raw_dir, filename)
        if not os.path.exists(path):
            raise FileNotFoundError(f"Missing file: {path}")
        logging.info(f"Loading {path} ...")
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        if isinstance(data, list):
            df = pd.DataFrame(data)
        elif isinstance(data, dict):
            df = pd.DataFrame.from_dict(data, orient='index').reset_index()
        else:
            raise ValueError(f"Unexpected JSON format in {filename}")
        logging.info(f"  Loaded {len(df)} rows | Columns: {list(df.columns)}")
        return df

    # ----------------------------------------------------------
    # Normalise columns
    # ----------------------------------------------------------
    def _normalise(self, df, source_tag):
        # Flexible text column detection
        for col in ['text', 'sentence', 'review', 'content', 'comment']:
            if col in df.columns:
                df = df.rename(columns={col: 'text'})
                break

        # Flexible label column detection
        for col in ['sentiment', 'polarity', 'label', 'opinion', 'category']:
            if col in df.columns:
                df = df.rename(columns={col: 'sentiment'})
                break

        if 'text' not in df.columns:
            raise ValueError(f"Cannot find text column in {source_tag}. Columns: {list(df.columns)}")
        if 'sentiment' not in df.columns:
            raise ValueError(f"Cannot find sentiment column in {source_tag}. Columns: {list(df.columns)}")

        df = df[['text', 'sentiment']].copy()
        df['text']      = df['text'].astype(str).str.strip()
        df['sentiment'] = df['sentiment'].astype(str).str.strip().str.title()
        df['source']    = source_tag
        return df

    # ----------------------------------------------------------
    # Clean
    # ----------------------------------------------------------
    def _clean(self, df):
        before = len(df)
        df = df.dropna(subset=['text', 'sentiment'])
        df = df[df['text'].str.len() > 2]
        df = df[df['sentiment'].isin(config.LABEL_MAPPING.keys())]
        df = df.drop_duplicates(subset=['text'])
        after = len(df)
        logging.info(f"  Cleaned: {before} → {after} rows (removed {before - after})")
        return df.reset_index(drop=True)

    # ----------------------------------------------------------
    # Simple augmentation (synonym swap placeholder)
    # ----------------------------------------------------------
    def _augment(self, df):
        """
        Lightweight augmentation: duplicate minority classes
        to reduce class imbalance. Replace with EDA/back-translation
        if needed.
        """
        counts    = df['sentiment'].value_counts()
        max_count = counts.max()
        augmented = [df]

        for label, count in counts.items():
            if count < max_count:
                minority = df[df['sentiment'] == label]
                needed   = max_count - count
                oversampled = minority.sample(n=needed, replace=True, random_state=config.RANDOM_SEED)
                augmented.append(oversampled)
                logging.info(f"  Augmented '{label}': +{needed} samples")

        df_aug = pd.concat(augmented, ignore_index=True).sample(
            frac=1, random_state=config.RANDOM_SEED
        ).reset_index(drop=True)
        logging.info(f"  After augmentation: {len(df_aug)} rows")
        return df_aug

    # ----------------------------------------------------------
    # Master runner
    # ----------------------------------------------------------
    def run(self):
        config.set_global_seed()

        # ── Load ───────────────────────────────────────────
        df_trainval = self._load_json(config.TRAIN_VAL_FILE)
        df_devtest  = self._load_json(config.DEV_TEST_FILE)

        df_trainval = self._normalise(df_trainval, 'train_val')
        df_devtest  = self._normalise(df_devtest,  'dev_test')

        # ── Clean ──────────────────────────────────────────
        logging.info("Cleaning train_val ...")
        df_trainval = self._clean(df_trainval)

        logging.info("Cleaning dev_test ...")
        df_devtest  = self._clean(df_devtest)

        # ── Split train_val → train / val ──────────────────
        logging.info("Splitting train_val → 80% train / 20% val ...")
        df_train, df_val = train_test_split(
            df_trainval,
            test_size=0.2,
            random_state=config.RANDOM_SEED,
            stratify=df_trainval['sentiment']
        )

        # ── Split dev_test → dev(ignored) / test ──────────
        logging.info("Splitting dev_test → 50% dev / 50% test ...")
        _, df_test = train_test_split(
            df_devtest,
            test_size=0.5,
            random_state=config.RANDOM_SEED,
            stratify=df_devtest['sentiment']
        )

        # ── Augment train only ─────────────────────────────
        logging.info("Augmenting train set ...")
        df_train_aug = self._augment(df_train)

        # ── Save ───────────────────────────────────────────
        df_train_aug.to_csv(config.TRAIN_AUG_PATH,  index=False)
        df_val.to_csv(config.VAL_CLEAN_PATH,         index=False)
        df_test.to_csv(config.TEST_CLEAN_PATH,        index=False)

        logging.info("\n" + "="*60)
        logging.info("✅ DATA PIPELINE COMPLETE")
        logging.info("="*60)
        logging.info(f"  Train (augmented) : {len(df_train_aug)} rows → {config.TRAIN_AUG_PATH}")
        logging.info(f"  Val               : {len(df_val)} rows → {config.VAL_CLEAN_PATH}")
        logging.info(f"  Test              : {len(df_test)} rows → {config.TEST_CLEAN_PATH}")
        logging.info("\nLabel distributions:")
        logging.info(f"  Train:\n{df_train_aug['sentiment'].value_counts().to_string()}")
        logging.info(f"  Val:\n{df_val['sentiment'].value_counts().to_string()}")
        logging.info(f"  Test:\n{df_test['sentiment'].value_counts().to_string()}")
        logging.info("➡️  Next: run 02_model_selection.py")


if __name__ == "__main__":
    try:
        pipeline = DataPipeline()
        pipeline.run()
    except Exception as e:
        logging.error(f"Data pipeline failed: {e}", exc_info=True)

Writing /content/01_data_pipeline.py


In [ ]:
# ── Stage 2 ────────────────────────────────────────────────
%%writefile /content/02_model_selection.py
# [paste your full 02_model_selection.py code here]
%%writefile /content/02_model_selection.py
import os
import sys
import logging
import importlib
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report

sys.path.insert(0, '/content')
if 'config' in sys.modules:
    importlib.reload(sys.modules['config'])
import config

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")


class SentimentDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts     = df['text'].tolist()
        self.labels    = df[config.TARGET_COLUMN].map(config.LABEL_MAPPING).tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


class ModelTrainer:

    def __init__(self, model_name, device):
        self.model_name = model_name
        self.device     = device

    def train_and_evaluate(self, df_train, df_val):
        tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        model     = AutoModelForSequenceClassification.from_pretrained(
            self.model_name, num_labels=len(config.LABEL_MAPPING)
        ).to(self.device)

        train_loader = DataLoader(
            SentimentDataset(df_train, tokenizer, config.MAX_SEQ_LENGTH),
            batch_size=config.BATCH_SIZE, shuffle=True
        )
        val_loader = DataLoader(
            SentimentDataset(df_val, tokenizer, config.MAX_SEQ_LENGTH),
            batch_size=config.BATCH_SIZE, shuffle=False
        )

        optimizer   = torch.optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
        total_steps = len(train_loader) * config.NUM_EPOCHS
        scheduler   = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
        criterion   = nn.CrossEntropyLoss()

        best_val_f1 = 0.0
        for epoch in range(1, config.NUM_EPOCHS + 1):
            model.train()
            total_loss = 0
            for batch in train_loader:
                input_ids      = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels         = batch['label'].to(self.device)
                optimizer.zero_grad()
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss    = criterion(outputs.logits, labels)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                total_loss += loss.item()

            model.eval()
            all_preds, all_labels = [], []
            with torch.no_grad():
                for batch in val_loader:
                    input_ids      = batch['input_ids'].to(self.device)
                    attention_mask = batch['attention_mask'].to(self.device)
                    labels         = batch['label'].to(self.device)
                    outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
                    preds          = torch.argmax(outputs.logits, dim=1)
                    all_preds.extend(preds.cpu().numpy())
                    all_labels.extend(labels.cpu().numpy())

            val_f1   = f1_score(all_labels, all_preds, average='macro', zero_division=0)
            avg_loss = total_loss / len(train_loader)
            logging.info(f"  [{self.model_name}] Epoch {epoch}/{config.NUM_EPOCHS} | Loss: {avg_loss:.4f} | Val F1: {val_f1:.4f}")

            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                save_dir    = os.path.join(config.MODEL_DIR, self.model_name.replace('/', '_'))
                os.makedirs(save_dir, exist_ok=True)
                model.save_pretrained(save_dir)
                tokenizer.save_pretrained(save_dir)

        report = classification_report(
            all_labels, all_preds,
            target_names=list(config.LABEL_MAPPING.keys()),
            zero_division=0
        )
        logging.info(f"\n{report}")
        del model
        torch.cuda.empty_cache()
        return best_val_f1


class ModelSelector:

    def run(self):
        config.set_global_seed()
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        logging.info(f"Device: {device}")

        if not os.path.exists(config.TRAIN_AUG_PATH) or not os.path.exists(config.VAL_CLEAN_PATH):
            raise FileNotFoundError("Missing processed data — run 01_data_pipeline.py first!")

        df_train = pd.read_csv(config.TRAIN_AUG_PATH)
        df_val   = pd.read_csv(config.VAL_CLEAN_PATH)
        logging.info(f"Train: {len(df_train)} | Val: {len(df_val)}")

        results     = []
        best_f1     = 0.0
        best_model  = ''

        for model_name in config.CANDIDATE_MODELS:
            logging.info(f"\n{'='*60}\nTraining: {model_name}\n{'='*60}")
            trainer  = ModelTrainer(model_name, device)
            val_f1   = trainer.train_and_evaluate(df_train, df_val)
            results.append({'model': model_name, 'val_f1_macro': round(val_f1, 4)})
            if val_f1 > best_f1:
                best_f1    = val_f1
                best_model = model_name

        df_results = pd.DataFrame(results).sort_values('val_f1_macro', ascending=False).reset_index(drop=True)
        results_path = os.path.join(config.RESULTS_DIR, 'model_selection_results.csv')
        df_results.to_csv(results_path, index=False)

        logging.info("\n" + "="*60)
        logging.info("📊 MODEL SELECTION RESULTS")
        logging.info("="*60)
        logging.info(f"\n{df_results.to_string(index=False)}")
        logging.info(f"\n🏆 Best model: {best_model} | Val F1 Macro: {best_f1:.4f}")
        logging.info(f"✅ Results saved → {results_path}")
        logging.info("⚠️  Update SELECTED_MODEL in config.py before running Stage 3!")
        logging.info("➡️  Next: update config.py → run 03_hyperparameter_tuning.py")


if __name__ == "__main__":
    try:
        selector = ModelSelector()
        selector.run()
    except Exception as e:
        logging.error(f"Model selection failed: {e}", exc_info=True)

Writing /content/02_model_selection.py


In [ ]:
# ── Stage 3 ────────────────────────────────────────────────
%%writefile /content/03_hyperparameter_tuning.py
# [paste your full 03_hyperparameter_tuning.py code here]
%%writefile /content/03_hyperparameter_tuning.py
import os
import sys
import logging
import importlib
import itertools
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
import json

sys.path.insert(0, '/content')
if 'config' in sys.modules:
    importlib.reload(sys.modules['config'])
import config

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")


class SentimentDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts     = df['text'].tolist()
        self.labels    = df[config.TARGET_COLUMN].map(config.LABEL_MAPPING).tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


class HyperparameterTuner:

    def __init__(self):
        self.device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model_name = config.SELECTED_MODEL
        logging.info(f"Device     : {self.device}")
        logging.info(f"Base model : {self.model_name}")

    def _train_fold(self, model, train_loader, val_loader, lr, weight_decay, num_epochs):
        optimizer   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        total_steps = len(train_loader) * num_epochs
        scheduler   = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
        criterion   = nn.CrossEntropyLoss()
        best_f1     = 0.0

        for epoch in range(1, num_epochs + 1):
            model.train()
            total_loss = 0
            for batch in train_loader:
                input_ids      = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels         = batch['label'].to(self.device)
                optimizer.zero_grad()
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss    = criterion(outputs.logits, labels)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                total_loss += loss.item()

            model.eval()
            all_preds, all_labels = [], []
            with torch.no_grad():
                for batch in val_loader:
                    input_ids      = batch['input_ids'].to(self.device)
                    attention_mask = batch['attention_mask'].to(self.device)
                    labels         = batch['label'].to(self.device)
                    outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
                    preds          = torch.argmax(outputs.logits, dim=1)
                    all_preds.extend(preds.cpu().numpy())
                    all_labels.extend(labels.cpu().numpy())

            fold_f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
            avg_loss = total_loss / len(train_loader)
            logging.info(f"    Epoch {epoch}/{num_epochs} | Loss: {avg_loss:.4f} | Val F1: {fold_f1:.4f}")
            if fold_f1 > best_f1:
                best_f1 = fold_f1

        return best_f1

    def _kfold_cv(self, df_train, tokenizer, lr, batch_size, weight_decay):
        dataset = SentimentDataset(df_train, tokenizer, config.MAX_SEQ_LENGTH)
        labels  = df_train[config.TARGET_COLUMN].map(config.LABEL_MAPPING).tolist()
        skf     = StratifiedKFold(n_splits=config.N_FOLDS, shuffle=True, random_state=config.RANDOM_SEED)
        fold_f1_scores = []

        for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels), 1):
            logging.info(f"  Fold {fold}/{config.N_FOLDS}")
            train_loader = DataLoader(dataset, batch_size=batch_size, sampler=SubsetRandomSampler(train_idx))
            val_loader   = DataLoader(dataset, batch_size=batch_size, sampler=SubsetRandomSampler(val_idx))
            model        = AutoModelForSequenceClassification.from_pretrained(
                self.model_name, num_labels=len(config.LABEL_MAPPING)
            ).to(self.device)
            fold_f1 = self._train_fold(model, train_loader, val_loader, lr, weight_decay, config.NUM_EPOCHS)
            fold_f1_scores.append(fold_f1)
            del model
            torch.cuda.empty_cache()

        mean_f1 = float(np.mean(fold_f1_scores))
        std_f1  = float(np.std(fold_f1_scores))
        logging.info(f"  CV Result → Mean F1: {mean_f1:.4f} ± {std_f1:.4f}")
        return mean_f1, std_f1

    def _train_final_model(self, df_train, df_val, tokenizer, best_params):
        logging.info(f"\nTraining FINAL model with best params: {best_params}")
        train_loader = DataLoader(
            SentimentDataset(df_train, tokenizer, config.MAX_SEQ_LENGTH),
            batch_size=best_params['batch_size'], shuffle=True
        )
        val_loader = DataLoader(
            SentimentDataset(df_val, tokenizer, config.MAX_SEQ_LENGTH),
            batch_size=best_params['batch_size'], shuffle=False
        )
        model = AutoModelForSequenceClassification.from_pretrained(
            self.model_name, num_labels=len(config.LABEL_MAPPING)
        ).to(self.device)
        optimizer   = torch.optim.AdamW(model.parameters(), lr=best_params['learning_rate'], weight_decay=best_params['weight_decay'])
        total_steps = len(train_loader) * config.NUM_EPOCHS
        scheduler   = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
        criterion   = nn.CrossEntropyLoss()
        best_val_f1 = 0.0
        best_report = ""

        for epoch in range(1, config.NUM_EPOCHS + 1):
            model.train()
            total_loss = 0
            for batch in train_loader:
                input_ids      = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels         = batch['label'].to(self.device)
                optimizer.zero_grad()
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss    = criterion(outputs.logits, labels)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                total_loss += loss.item()

            model.eval()
            all_preds, all_labels = [], []
            with torch.no_grad():
                for batch in val_loader:
                    input_ids      = batch['input_ids'].to(self.device)
                    attention_mask = batch['attention_mask'].to(self.device)
                    labels         = batch['label'].to(self.device)
                    outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
                    preds          = torch.argmax(outputs.logits, dim=1)
                    all_preds.extend(preds.cpu().numpy())
                    all_labels.extend(labels.cpu().numpy())

            val_f1   = f1_score(all_labels, all_preds, average='macro', zero_division=0)
            avg_loss = total_loss / len(train_loader)
            report   = classification_report(all_labels, all_preds, target_names=list(config.LABEL_MAPPING.keys()), zero_division=0)
            logging.info(f"Epoch {epoch}/{config.NUM_EPOCHS} | Loss: {avg_loss:.4f} | Val F1: {val_f1:.4f}")

            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_report = report
                save_dir    = os.path.join(config.MODEL_DIR, 'best_model')
                os.makedirs(save_dir, exist_ok=True)
                model.save_pretrained(save_dir)
                tokenizer.save_pretrained(save_dir)
                logging.info(f"✅ Best model saved → {save_dir}")

        logging.info(f"\n🏆 Final Best Val F1: {best_val_f1:.4f}")
        logging.info(f"Classification Report:\n{best_report}")
        return best_val_f1, best_report

    def run(self):
        config.set_global_seed()

        if not os.path.exists(config.TRAIN_AUG_PATH):
            raise FileNotFoundError(f"Missing: {config.TRAIN_AUG_PATH} — run 01_data_pipeline.py first!")
        if not os.path.exists(config.VAL_CLEAN_PATH):
            raise FileNotFoundError(f"Missing: {config.VAL_CLEAN_PATH} — run 01_data_pipeline.py first!")

        df_train  = pd.read_csv(config.TRAIN_AUG_PATH)
        df_val    = pd.read_csv(config.VAL_CLEAN_PATH)
        tokenizer = AutoTokenizer.from_pretrained(self.model_name)

        param_grid = config.TUNING_PARAM_GRID
        keys       = list(param_grid.keys())
        combos     = list(itertools.product(*[param_grid[k] for k in keys]))
        logging.info(f"Total combinations: {len(combos)} | {config.N_FOLDS}-Fold CV each")

        tuning_results = []
        best_mean_f1   = 0.0
        best_params    = {}

        for i, combo in enumerate(combos, 1):
            params = dict(zip(keys, combo))
            logging.info(f"\n[Combo {i}/{len(combos)}] {params}")
            mean_f1, std_f1 = self._kfold_cv(df_train, tokenizer, params['learning_rate'], params['batch_size'], params['weight_decay'])
            tuning_results.append({**params, 'mean_cv_f1': round(mean_f1, 4), 'std_cv_f1': round(std_f1, 4)})
            if mean_f1 > best_mean_f1:
                best_mean_f1 = mean_f1
                best_params  = params.copy()

        df_tuning    = pd.DataFrame(tuning_results).sort_values('mean_cv_f1', ascending=False).reset_index(drop=True)
        tuning_path  = os.path.join(config.RESULTS_DIR, 'tuning_results.csv')
        df_tuning.to_csv(tuning_path, index=False)
        logging.info(f"\n📊 Tuning Results:\n{df_tuning.to_string(index=False)}")
        logging.info(f"\n🏆 Best params: {best_params} | Mean CV F1: {best_mean_f1:.4f}")

        best_params_path = os.path.join(config.RESULTS_DIR, 'best_params.json')
        with open(best_params_path, 'w') as f:
            json.dump(best_params, f, indent=2)

        final_val_f1, final_report = self._train_final_model(df_train, df_val, tokenizer, best_params)

        summary = {
            'model_name': self.model_name, 'best_params': best_params,
            'best_cv_f1': round(best_mean_f1, 4), 'final_val_f1': round(final_val_f1, 4),
            'final_val_report': final_report
        }
        with open(os.path.join(config.RESULTS_DIR, 'tuning_summary.json'), 'w') as f:
            json.dump(summary, f, indent=2)

        logging.info("🎉 Stage 3 complete. ➡️  Next: run 04_final_evaluation.py")


if __name__ == "__main__":
    try:
        tuner = HyperparameterTuner()
        tuner.run()
    except Exception as e:
        logging.error(f"Hyperparameter Tuning failed: {e}", exc_info=True)

Writing /content/03_hyperparameter_tuning.py


In [ ]:
# ── Stage 4 ────────────────────────────────────────────────
%%writefile /content/04_final_evaluation.py
# [paste your full 04_final_evaluation.py code here]
%%writefile /content/04_final_evaluation.py
import os
import sys
import logging
import importlib
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, '/content')
if 'config' in sys.modules:
    importlib.reload(sys.modules['config'])
import config

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")


class SentimentDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts     = df['text'].tolist()
        self.labels    = df[config.TARGET_COLUMN].map(config.LABEL_MAPPING).tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


class FinalEvaluator:

    def __init__(self):
        self.device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model_dir = os.path.join(config.MODEL_DIR, 'best_model')

    def _load_model(self):
        if not os.path.exists(self.model_dir):
            raise FileNotFoundError(f"Missing: {self.model_dir} — run 03_hyperparameter_tuning.py first!")
        tokenizer = AutoTokenizer.from_pretrained(self.model_dir)
        model     = AutoModelForSequenceClassification.from_pretrained(
            self.model_dir, num_labels=len(config.LABEL_MAPPING)
        ).to(self.device)
        model.eval()
        return model, tokenizer

    def _run_inference(self, model, loader):
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in loader:
                input_ids      = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels         = batch['label'].to(self.device)
                outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
                preds          = torch.argmax(outputs.logits, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        return all_labels, all_preds

    def _plot_confusion_matrix(self, all_labels, all_preds):
        label_names = list(config.LABEL_MAPPING.keys())
        cm          = confusion_matrix(all_labels, all_preds)
        cm_norm     = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        fig, axes   = plt.subplots(1, 2, figsize=(14, 5))
        sns.heatmap(cm,      annot=True, fmt='d',    cmap='Blues',   xticklabels=label_names, yticklabels=label_names, ax=axes[0])
        sns.heatmap(cm_norm, annot=True, fmt='.2%',  cmap='Oranges', xticklabels=label_names, yticklabels=label_names, ax=axes[1])
        axes[0].set_title('Confusion Matrix (Counts)')
        axes[1].set_title('Confusion Matrix (Normalised)')
        for ax in axes:
            ax.set_xlabel('Predicted')
            ax.set_ylabel('True')
        plt.suptitle(f'Final Test Evaluation — {config.SELECTED_MODEL}', fontweight='bold')
        plt.tight_layout()
        save_path = os.path.join(config.RESULTS_DIR, 'confusion_matrix.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        logging.info(f"✅ Confusion matrix saved → {save_path}")

    def _plot_per_class_f1(self, all_labels, all_preds):
        label_names = list(config.LABEL_MAPPING.keys())
        f1_scores   = f1_score(all_labels, all_preds, average=None, zero_division=0)
        colors      = ['#2196F3', '#F44336', '#FF9800']
        plt.figure(figsize=(7, 4))
        bars = plt.bar(label_names, f1_scores, color=colors, edgecolor='black', width=0.5)
        for bar, score in zip(bars, f1_scores):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.01, f'{score:.4f}', ha='center', fontweight='bold')
        plt.ylim(0, 1.1)
        plt.title(f'Per-Class F1 — Test Set\n{config.SELECTED_MODEL}', fontweight='bold')
        plt.xlabel('Sentiment Class')
        plt.ylabel('F1 Score')
        plt.axhline(y=0.5, color='red', linestyle='--', alpha=0.5)
        plt.tight_layout()
        save_path = os.path.join(config.RESULTS_DIR, 'per_class_f1.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        logging.info(f"✅ Per-class F1 chart saved → {save_path}")

    def _save_metrics(self, all_labels, all_preds):
        label_names   = list(config.LABEL_MAPPING.keys())
        f1_per_class  = f1_score(all_labels, all_preds, average=None,      zero_division=0)
        pre_per_class = precision_score(all_labels, all_preds, average=None, zero_division=0)
        rec_per_class = recall_score(all_labels, all_preds, average=None,   zero_division=0)
        f1_macro      = f1_score(all_labels, all_preds, average='macro',    zero_division=0)
        f1_weighted   = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
        accuracy      = accuracy_score(all_labels, all_preds)

        rows = [{'class': n, 'precision': round(pre_per_class[i], 4), 'recall': round(rec_per_class[i], 4), 'f1_score': round(f1_per_class[i], 4)} for i, n in enumerate(label_names)]
        rows += [
            {'class': '--- OVERALL ---', 'precision': '', 'recall': '', 'f1_score': ''},
            {'class': 'F1 Macro',        'precision': '', 'recall': '', 'f1_score': round(f1_macro, 4)},
            {'class': 'F1 Weighted',     'precision': '', 'recall': '', 'f1_score': round(f1_weighted, 4)},
            {'class': 'Accuracy',        'precision': '', 'recall': '', 'f1_score': round(accuracy, 4)}
        ]
        pd.DataFrame(rows).to_csv(config.FINAL_METRICS_PATH, index=False)
        logging.info(f"✅ Metrics saved → {config.FINAL_METRICS_PATH}")
        return f1_macro, f1_weighted, accuracy, f1_per_class

    def run(self):
        config.set_global_seed()

        flag_path = os.path.join(config.RESULTS_DIR, '.test_evaluated')
        if os.path.exists(flag_path):
            logging.warning("⛔ Test set already evaluated! Delete .test_evaluated only if absolutely necessary.")
            return

        if not os.path.exists(config.TEST_CLEAN_PATH):
            raise FileNotFoundError(f"Missing: {config.TEST_CLEAN_PATH} — run 01_data_pipeline.py first!")

        df_test          = pd.read_csv(config.TEST_CLEAN_PATH)
        model, tokenizer = self._load_model()
        test_loader      = DataLoader(SentimentDataset(df_test, tokenizer, config.MAX_SEQ_LENGTH), batch_size=config.BATCH_SIZE, shuffle=False)

        logging.info("🔍 Running inference on test set (ONE TIME ONLY)...")
        all_labels, all_preds = self._run_inference(model, test_loader)

        report = classification_report(all_labels, all_preds, target_names=list(config.LABEL_MAPPING.keys()), zero_division=0)
        logging.info(f"\n📊 FINAL TEST REPORT:\n{report}")

        f1_macro, f1_weighted, accuracy, f1_per_class = self._save_metrics(all_labels, all_preds)

        logging.info(f"\n{'='*60}\n🏆 FINAL TEST RESULTS\n{'='*60}")
        logging.info(f"  F1 Macro    : {f1_macro:.4f}")
        logging.info(f"  F1 Weighted : {f1_weighted:.4f}")
        logging.info(f"  Accuracy    : {accuracy:.4f}")

        self._plot_confusion_matrix(all_labels, all_preds)
        self._plot_per_class_f1(all_labels, all_preds)

        with open(flag_path, 'w') as f:
            f.write("Test set evaluated.")
        logging.info("🔒 Flag written. 🎉 Stage 4 complete. ➡️  Next: run 05_cross_domain_evaluation.py")


if __name__ == "__main__":
    try:
        evaluator = FinalEvaluator()
        evaluator.run()
    except Exception as e:
        logging.error(f"Final Evaluation failed: {e}", exc_info=True)

Writing /content/04_final_evaluation.py


In [ ]:
# ── Stage 5 ────────────────────────────────────────────────
%%writefile /content/05_cross_domain_evaluation.py
# [paste your full 05_cross_domain_evaluation.py code here]

Writing /content/05_cross_domain_evaluation.py


In [ ]:
def load_config():
    import importlib, sys
    if 'config' in sys.modules:
        del sys.modules['config']
    sys.path.insert(0, '/content')
    import config
    return config

config = load_config()
print("✅ Config loaded")

✅ config.py loaded
✅ Config loaded


In [ ]:
config = load_config()
exec(open('/content/01_data_pipeline.py').read())
pipeline = DataPipeline()
pipeline.run()

✅ config.py loaded


FileNotFoundError: [Errno 2] No such file or directory: '/content/01_data_pipeline.py'

In [ ]:
SELECTED_MODEL = 'roberta-base'   # ← replace with your winner

In [ ]:
# Read Stage 2 results
import pandas as pd
df = pd.read_csv('/content/results/model_selection_results.csv')
print(df.sort_values('val_f1_macro', ascending=False).to_string(index=False))

# ── Set winner manually ────────────────────────────────────
WINNER = 'roberta-base'   # ← change this to your best model

# Patch config.py
with open('/content/config.py', 'r') as f:
    content = f.read()

import re
content = re.sub(
    r"SELECTED_MODEL\s*=\s*'[^']+'",
    f"SELECTED_MODEL = '{WINNER}'",
    content
)
with open('/content/config.py', 'w') as f:
    f.write(content)

config = load_config()
print(f"\n✅ SELECTED_MODEL updated → {config.SELECTED_MODEL}")

In [ ]:
config = load_config()
exec(open('/content/03_hyperparameter_tuning.py').read())
tuner = HyperparameterTuner()
tuner.run()

In [ ]:
#Cell 9 — ▶️ Stage 4: Final Evaluation (ONE TIME ONLY):
config = load_config()
exec(open('/content/04_final_evaluation.py').read())
evaluator = FinalEvaluator()
evaluator.run()

In [ ]:
# Optional: upload SemEval / MAMS first
# from google.colab import files
# files.upload()

config = load_config()
exec(open('/content/05_cross_domain_evaluation.py').read())
cross = CrossDomainEvaluator()
cross.run()

In [ ]:
import pandas as pd, os

print("="*60)
print("📊 PIPELINE RESULTS SUMMARY")
print("="*60)

# Model selection
ms_path = '/content/results/model_selection_results.csv'
if os.path.exists(ms_path):
    print("\n🔹 Stage 2 — Model Selection:")
    print(pd.read_csv(ms_path).sort_values('val_f1_macro', ascending=False).to_string(index=False))

# Tuning results
tr_path = '/content/results/tuning_results.csv'
if os.path.exists(tr_path):
    print("\n🔹 Stage 3 — Tuning Results (top 3):")
    print(pd.read_csv(tr_path).head(3).to_string(index=False))

# Final metrics
fm_path = '/content/results/final_summary_metrics.csv'
if os.path.exists(fm_path):
    print("\n🔹 Stage 4 — Final Test Metrics:")
    print(pd.read_csv(fm_path).to_string(index=False))

# Cross-domain
cd_path = '/content/results/cross_domain_results.csv'
if os.path.exists(cd_path):
    print("\n🔹 Stage 5 — Cross-Domain Results:")
    print(pd.read_csv(cd_path).to_string(index=False))

print("\n" + "="*60)
print("✅ All done!")

In [ ]:
import importlib, sys, os

# Force clean config load
if 'config' in sys.modules:
    del sys.modules['config']
sys.path.insert(0, '/content')
import config

print("✅ Config loaded:", config.PROCESSED_DATA_DIR)

In [ ]:
exec(open('/content/01_data_pipeline.py').read())
pipeline = DataPipelineBuilder()
pipeline.run_pipeline()

In [ ]:
exec(open('/content/02_model_selection.py').read())
selector = ModelSelector()
selector.run()

In [ ]:
exec(open('/content/03_hyperparameter_tuning.py').read())
tuner = HyperparameterTuner()
tuner.run()

In [ ]:
exec(open('/content/04_final_evaluation.py').read())
evaluator = FinalEvaluator()
evaluator.run()

In [ ]:
exec(open('/content/05_cross_domain_evaluation.py').read())
cross = CrossDomainEvaluator()
cross.run()